# EP2 - IT Assist: Multi-Agent Orchestration
## 3 Agentes con CrewAI sobre Base RAG de EP1

Usuario -> Investigador -> Analista -> Comunicador -> Usuario
| Agente | Rol | Tools |
|--------|-----|-------|
| Investigador | Busca en FAISS | buscar_base_conocimiento_it |
| Analista | Sintetiza | clasificar_prioridad_ticket |
| Comunicador | Redacta | (sin tools) |

## 2. Setup
Verifica Ollama + Groq API Key.

In [ ]:
import os, sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))
from dotenv import load_dotenv
load_dotenv()
print("LLM_MODEL:", os.getenv("LLM_MODEL", "openai/qwen/qwen3.8-27b"))
print("API_KEY:", "***" if os.getenv("LLM_API_KEY") else "FALTA")

## 3. Base Vectorial (EP1)
Verifica FAISS cargado.

In [ ]:
from rag_tool import _get_engine, buscar_base_conocimiento
engine = _get_engine()
print(f"FAISS: {engine.index.ntotal} vectores, {engine.index.d} dims")

## 4. RAG Tool (Agente Investigador)

In [ ]:
resultado = buscar_base_conocimiento("Olvide mi contrasena corporativa")
print(resultado[:500])

## 5. Pipeline Secuencial (3 Agentes)
Investigador -> Analista -> Comunicador

In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
def _llm():
    return dict(model=os.getenv("LLM_MODEL"), api_key=os.getenv("LLM_API_KEY"), base_url=os.getenv("LLM_BASE_URL"), temperature=0.3)
@tool("buscar_base_conocimiento_it")
def buscar(consulta: str) -> str:
    """Busca en la base IT."""
    return buscar_base_conocimiento(consulta)
ag_inv = Agent(role="Investigador IT", goal="Encontrar fragmentos.", backstory="Especialista RAG.", tools=[buscar], llm=_llm(), allow_delegation=False, max_iter=4)
ag_an = Agent(role="Analista IT", goal="Sintetizar.", backstory="Analista TI.", tools=[], llm=_llm(), allow_delegation=False, max_iter=4)
ag_co = Agent(role="Comunicador IT", goal="Redactar.", backstory="Redactor.", tools=[], llm=_llm(), allow_delegation=False, max_iter=3)
print("3 Agentes OK")

In [ ]:
t_inv = Task(description="Busca en base.", expected_output="Fragmentos.", agent=ag_inv)
t_an = Task(description="Sintetiza.", expected_output="Analisis.", agent=ag_an, context=[t_inv])
t_co = Task(description="Redacta.", expected_output="Respuesta.", agent=ag_co, context=[t_inv, t_an])
crew = Crew(agents=[ag_inv, ag_an, ag_co], tasks=[t_inv, t_an, t_co], process=Process.sequential, verbose=True)
result = crew.kickoff()
print(result.raw)

## 6. Pipeline Jerarquico
Manager evalua y re-delega.

In [ ]:
crew_hier = Crew(agents=[ag_inv, ag_an, ag_co], tasks=[t_inv, t_an, t_co], process=Process.hierarchical, manager_llm=_llm(), verbose=True)
print("Hierarchical OK")

## 7. Memoria
Short-term: context=[t_inv] | Long-term: memory=True
Ver 3-memory_short_long.ipynb

## 8. Streamlit App
Ejecutar: streamlit run app_multiagent.py
Tabs: Consultas | Trazabilidad | Testeo

## 9. Conclusiones
1. CrewAI orquesta agentes sin codear flujo.
2. Sequential vs Hierarchical = lineal vs manager.
3. Memoria mejora multi-turno.
4. Trazabilidad por agente = auditoria.
5. FAISS EP1 reutilizado sin cambios.